In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
class DecisionMaker:

    def __init__(self, n_criteria, n_pieces):
        self.n_criteria = n_criteria
        self.n_pieces = n_pieces

        self.slopes = self.build_random_decision_function()
        self.breakpoints_x = np.linspace(0, self.n_pieces, self.n_pieces+1)
        self.break_point_y = np.stack([[0] + [np.sum(self.slopes[i][:j+1]) for j in range(self.n_pieces)] for i in range(self.n_criteria)])

    def build_random_decision_function(self):
        slopes = []
        for i in range(self.n_criteria):
            crit_slopes = []
            for j in range(self.n_pieces):
                if i == 0 and j == 0:
                    crit_slopes.append(1.)
                else:
                    crit_slopes.append(np.random.uniform(0, 10))
            slopes.append(crit_slopes)
        return np.stack(slopes)

    def plot_decision_function(self):
        x = np.linspace(0, self.n_pieces, self.n_pieces+1)
        for i in range(self.n_criteria):
            y = [0]
            for j in range(self.n_pieces):
                y.append(y[-1] + self.slopes[i][j])
            plt.plot(x, y, label=f'Criterion {i+1}')
        plt.legend()
        plt.show()

    def get_ui(self, criterion, value):
        marginal_utility = 0
        for i in range(self.n_pieces):
            sub_value = np.max([np.min([1., value - i]), 0.]) * self.slopes[criterion][i]
            marginal_utility += sub_value
        return marginal_utility

    def answer(self, criterion_i, criterion_j, q_i, p_i, q_j):
        du = self.get_ui(criterion_i, p_i) - self.get_ui(criterion_i, q_i)
        u_j = self.get_ui(criterion_j, q_j)

        if du > 0:
            if du > u_j:
                return None
            else:
                for i in range(self.n_pieces):
                    if u_j - self.break_point_y[criterion_j][i] >= du and u_j - self.break_point_y[criterion_j][i+1] < du:
                        
                        dv = du - u_j + self.break_point_y[criterion_j][i+1]
                        return i+1 - (dv / self.slopes[criterion_j][i])
        else:
            if -du > (self.break_point_y[criterion_j][-1] - u_j):
                return None
            else:
                for i in range(self.n_pieces):
                    if self.break_point_y[criterion_j][i+1] - u_j > -du and self.break_point_y[criterion_j][i] - u_j <= -du:
                        
                        dv = -du + u_j - self.break_point_y[criterion_j][i]
                        return i + dv / self.slopes[criterion_j][i]
        

In [ ]:
dm = DecisionMaker(2,5)
dm.plot_decision_function()

In [ ]:
class HiddenDecisionMakers:

    def __init__(self, n_dms, n_criteria, n_pieces):
        self.dms = [DecisionMaker(n_criteria, n_pieces) for i in range(n_dms)]

    def answer(self, criterion_i, criterion_j, q_i, p_i, q_j):
        answers = [dm.answer(criterion_i, criterion_j, q_i, p_i, q_j) for dm in self.dms]
        return np.random.permutation(answers)


class SquaredQuery:

    def __init__(self, criterion_i, criterion_j, min_i, max_i, min_j, max_j):
        self.criterion_i = criterion_i
        self.criterion_j = criterion_j
        self.min_i = min_i
        self.max_i = max_i
        self.min_j = min_j
        self.max_j = max_j

    def query_hidden_dms(self, hdm):
        # Q1
        # crit_i, crit_j, q_i, p_i, q_j

        q_i = self.max_i
        p_i = self.min_i
        q_j = self.min_j

        answer_1 = hdm.answer(criterion_i=self.criterion_i, criterion_j=self.criterion_j, q_i=q_i, q_j=q_j, p_i=p_i)
        if np.sum(answer_1 > self.max_j) == 2:
            answer_1 = hdm.answer(criterion_i=self.criterion_j, criterion_j=self.criterion_i, q_i=self.max_j, q_j=self.min_i, p_i=self.min_j)
            assert None not in answer_1
            return [self.criterion_i, self.criterion_j], [[self.min_i, self.max_j], [answer_1[0], self.min_j],], [[self.min_i, self.max_j], [answer_1[1], self.min_j]]

        elif np.sum(answer_1 > self.max_j) == 1:
            q_i = np.min(answer_1)
            p_i = self.min_i
            q_j = self.min_j
            answer_1 = hdm.answer(criterion_i=self.criterion_j, criterion_j=self.criterion_i, q_i=q_i, q_j=q_j, p_i=p_i)
            assert None not in answer_1
            return [self.criterion_i, self.riterion_j], [[self.min_i, self.max_j], [answer_1[0], self.min_j],], [[self.min_i, self.max_j], [answer_1[1], self.min_j]]

        else:

            return [self.criterion_i, self.criterion_j], [[self.max_i, self.min_j], [self.min_i, answer_1[0]]], [[self.max_i, self.min_j], [self.min_i, answer_1[1]]]

    
class BridgeQuery:

    def __init__(self, bridged_criterion, constrained_criterion, bridged_breakpoint, min_bridged, max_bridged, min_squared, max_squared):
        self.bridged_criterion = bridged_criterion
        self.constrained_criterion = constrained_criterion

        # maybe do better
        self.bridged_breakpoint = bridged_breakpoint
        self.min_bridged = min_bridged
        self.max_bridged = max_bridged
        self.min_squared = min_squared
        self.max_squared = max_squared

    def query_hidden_dms(self, hdm):
        # Q1
        # crit_i, crit_j, q_i, p_i, q_j

        q_i = self.min_squared + (self.max_squared - self.min_squared) / 2
        p_i = self.min_squared
        q_j = (self.bridged_breakpoint - self.min_bridged) / 2 + self.min_bridged

        answer_1 = hdm.answer(criterion_i=self.constrained_criterion, criterion_j=self.bridged_criterion, q_i=q_i, q_j=q_j, p_i=p_i)

        while True:
            if np.sum(answer_1 > self.bridged_breakpoint) == 2 and np.sum(answer_1 < self.max_bridged) == 2:
                return ([q_i, q_j], [p_i, answer_1[0]]), ([q_i, q_j], [p_i, answer_1[1]])

            elif np.sum(answer_1 > self.bridged_breakpoint) < 2:
                q_j = q_j + (self.bridged_breakpoint - np.min(answer_1)) + (np.min(answer_1) - q_j) / 2
                answer_1 = hdm.answer(criterion_i=self.constrained_criterion, criterion_j=self.bridged_criterion, q_i=q_i, q_j=q_j, p_i=p_i)

            else:
                if np.max(answer_1) - np.min(answer_1) < self.max_bridged - self.bridged_breakpoint:
                    q_j = q_j + (self.max_bridged - np.max(answer_1))
                    answer_1 = hdm.answer(criterion_i=self.constrained_criterion, criterion_j=self.bridged_criterion, q_i=q_i, q_j=q_j, p_i=p_i)
                else:
                    q_i = q_i - (q_i - self.min_squared) / 2
                    answer_1 = hdm.answer(criterion_i=self.constrained_criterion, criterion_j=self.bridged_criterion, q_i=q_i, q_j=q_j, p_i=p_i)


In [ ]:
class UTAsIdentification:

    def __init__(self, n_criteria, n_pieces):
        self.n_criteria = n_criteria
        self.n_pieces = n_pieces
        self.slopes = {
            "alpha": [
                [] for _ in range(n_criteria)
            ],
            "beta": [
                [] for _ in range(n_criteria)
            ]
        }
        self.slopes["alpha"][0].append(1.)
        self.slopes["beta"][0].append(1.)

    def identify_21(self, hdms):
        query = SquaredQuery(criterion_i=0, criterion_j=1, min_i=0, max_i=1, min_j=0, max_j=1)
        crits, I1, I2 = query.query_hidden_dms(hdm)

        self.slopes["alpha"][1].append((I1[0][0] - I1[1][0]) / (I1[1][1] - I1[0][1]))
        self.slopes["beta"][1].append((I2[0][0] - I2[1][0]) / (I2[1][1] - I2[0][1]))

    def identify_22(self, hdms):
        query_1 = SquaredQuery(criterion_i=0, criterion_j=1, min_i=0, max_i=1, min_j=1, max_j=2)
        crits_1, I1_1, I2_1 = query_1.query_hidden_dms(hdm)

        query_2 = BridgeQuery(bridged_criterion=1, 
        constrained_criterion=0,
        bridged_breakpoint=1, 
        min_bridged=0, 
        max_bridged=2, 
        min_squared=0, 
        max_squared=1)

        I1_2, I2_2 = query_2.query_hidden_dms(hdm)
        
        unidentified_slope_1 = (I1_1[0][0] - I1_1[1][0]) / (I1_1[1][1] - I1_1[0][1])
        unidentified_slope_2 = (I2_1[0][0] - I2_1[1][0]) / (I2_1[1][1] - I2_1[0][1])

        fixed_q = (I1_2[1][0] - I1_2[0][0]) / (I1_2[0][1] - 1)
        u_quotient_21 = (I1_2[1][1] - 1) / (I1_2[0][1] - 1)
        u_quotient_22 = (I2_2[1][1] - 1) / (I2_2[0][1] - 1)

        fixed_q_1 = (I1_2[0][0] - I1_2[1][0]) / (I1_2[1][1] - 1)
        fixed_q_2 = (I2_2[0][0] - I2_2[1][0]) / (I2_2[1][1] - 1)
        u_quotient_21 = (I1_2[0][1] - 1) / (I1_2[1][1] - 1)
        u_quotient_22 = (I2_2[0][1] - 1) / (I2_2[1][1] - 1)

        possible_slopes_1 = [
            fixed_q_1 + u_quotient_21 * self.slopes["alpha"][1][0],
            fixed_q_2 + u_quotient_22 * self.slopes["beta"][1][0],
        ]
        possible_slopes_2 = [
            fixed_q_2 + u_quotient_22 * self.slopes["alpha"][1][0],
            fixed_q_1 + u_quotient_21 * self.slopes["beta"][1][0],
        ]

        case_1 = np.min([
            np.abs(possible_slopes_1[0] -unidentified_slope_1) + np.abs(possible_slopes_1[1] -unidentified_slope_2), 
            np.abs(possible_slopes_1[1] -unidentified_slope_1) + np.abs(possible_slopes_1[0] -unidentified_slope_2)
        ])

        case_2 = np.min([
            np.abs(possible_slopes_2[0] -unidentified_slope_1) + np.abs(possible_slopes_2[1] -unidentified_slope_2), 
            np.abs(possible_slopes_2[1] -unidentified_slope_1) + np.abs(possible_slopes_2[0] -unidentified_slope_2)
        ])

        if case_1 < case_2:
            self.slopes["alpha"][1].append(possible_slopes_1[0])
            self.slopes["beta"][1].append(possible_slopes_1[1])

        else:
            self.slopes["alpha"][1].append(possible_slopes_2[0])
            self.slopes["beta"][1].append(possible_slopes_2[1])

    def generic_identification(self, hdms, criterion, square):
        if criterion == 0:
            opposite_criterion = 1
        else:
            opposite_criterion = 0

        query = SquaredQuery(criterion_i=criterion,
        criterion_j=opposite_criterion,
        min_i=square,
        max_i=square+1,
        min_j=0,
        max_j=1)
        crits, I1, I2 = query.query_hidden_dms(hdm)

        if opposite_criterion == 0:
            slopes_values = [(I1[0][0] - I1[1][0]) / (I1[1][1] - I1[0][1]), ((I2[0][0] - I2[1][0]) / (I2[1][1] - I2[0][1]))]
        else:
            slopes_values = [[((I1[1][1] - I1[0][1]) / (I1[0][0] - I1[1][0])) * self.slopes["alpha"][opposite_criterion][0], 
            ((I2[1][1] - I2[0][1]) / (I2[0][0] - I2[1][0])) * self.slopes["beta"][opposite_criterion][0]],
            [((I2[1][1] - I2[0][1]) / (I2[0][0] - I2[1][0])) * self.slopes["alpha"][opposite_criterion][0], 
            ((I1[1][1] - I1[0][1]) / (I1[0][0] - I1[1][0])) * self.slopes["beta"][opposite_criterion][0]],]
        
        query = SquaredQuery(criterion_i=criterion,
        criterion_j=opposite_criterion,
        min_i=square,
        max_i=square+1,
        min_j=1,
        max_j=2)
        crits, I1, I2 = query.query_hidden_dms(hdm)

        possible_slopes = [((I1[1][1] - I1[0][1]) / (I1[0][0] - I1[1][0])) * self.slopes["alpha"][opposite_criterion][1], 
        ((I2[1][1] - I2[0][1]) / (I2[0][0] - I2[1][0]))* self.slopes["beta"][opposite_criterion][1]]
        other_possible_slopes = [((I1[1][1] - I1[0][1]) / (I1[0][0] - I1[1][0])) * self.slopes["beta"][opposite_criterion][1], 
        ((I2[1][1] - I2[0][1]) / (I2[0][0] - I2[1][0]))* self.slopes["alpha"][opposite_criterion][1]]

        if opposite_criterion == 0:
            case_1 = np.min([
                np.abs(possible_slopes[0] - slopes_values[0]) + np.abs(possible_slopes[1] - slopes_values[1]),
                np.abs(possible_slopes[1] - slopes_values[0]) + np.abs(possible_slopes[0] - slopes_values[1])
            ])

            case_2 = np.min([
                np.abs(other_possible_slopes[0] - slopes_values[0]) + np.abs(other_possible_slopes[1] - slopes_values[1]),
                np.abs(other_possible_slopes[1] - slopes_values[0]) + np.abs(other_possible_slopes[0] - slopes_values[1])
            ])
        else:
            case_1 = np.min([
                np.abs(possible_slopes[0] - slopes_values[0][0]) + np.abs(possible_slopes[1] - slopes_values[0][1]),
                np.abs(possible_slopes[0] - slopes_values[1][0]) + np.abs(possible_slopes[0] - slopes_values[1][1])
            ])

            case_2 = np.min([
                np.abs(other_possible_slopes[0] - slopes_values[0][0]) + np.abs(other_possible_slopes[1] - slopes_values[0][1]),
                np.abs(other_possible_slopes[1] - slopes_values[0][1]) + np.abs(other_possible_slopes[0] - slopes_values[1][1])
            ])
            
        if case_1 < case_2:
            self.slopes["alpha"][0].append(possible_slopes[0])
            self.slopes["beta"][0].append(possible_slopes[1])
        else:
            self.slopes["alpha"][0].append(other_possible_slopes[0])
            self.slopes["beta"][0].append(other_possible_slopes[1])


    def identify_hdms(self, hdms):
        self.identify_21(hdm)
        self.identify_22(hdm)
        self.generic_identification(hdms=hdm, criterion=0, square=1)

        for i in range(2, self.n_pieces):
            self.generic_identification(hdms=hdm, criterion=0, square=i)
            self.generic_identification(hdms=hdm, criterion=1, square=i)
            
        for i in range(2, self.n_criteria):
            for j in range(self.n_pieces):
                self.generic_identification(hdms=hdm, criterion=i, square=j)

In [ ]:
identif = UTAsIdentification(2, 2)

hdm = HiddenDecisionMakers(n_dms=2, n_criteria=2, n_pieces=5)
identif.identify_hdms(hdm)

In [ ]:
identif.slopes

In [ ]:
hdm.dms[0].slopes, hdm.dms[1].slopes

In [ ]:
query_2 = BridgeQuery(bridged_criterion=1, 
        constrained_criterion=0,
        bridged_breakpoint=1, 
        min_bridged=0, 
        max_bridged=2, 
        min_squared=0, 
        max_squared=1)
I1_2, I2_2 = query_2.query_hidden_dms(hdm)

In [ ]:
 I2_2

In [ ]:
hdm = HiddenDecisionMakers(2, 2, 2)

hdm.dms[0].plot_decision_function()
hdm.dms[1].plot_decision_function()

In [ ]:
query = SquaredQuery(criterion_i=0, criterion_j=1, min_i=0, max_i=1, min_j=0, max_j=1)
crits, I1, I2 = query.query_hidden_dms(hdm)

slope_1 = (Q[1] - Q[0]) / (A_1[0]- A_1[1])
slope_2 = (Q[1] - Q[0]) / (A_2[0]- A_2[1])

slope_1, slope_2

In [ ]:
(I1[0][0] - I1[0][1]) / (I1[1][1] - I1[1][0])

In [ ]:
hdm.dms[0].slopes

In [ ]:
slopes_alpha = [[1.], []]
slopes_beta = [[1.], []]

# crit_i, crit_j, q_i, p_i, q_j

q_i = 1
p_i = 0
q_j = 0

answer_1 = hdm.answer(criterion_i=0, criterion_j=1, q_i=q_i, q_j=q_j, p_i=p_i)
print(answer_1)

if np.sum(answer_1 > 1) == 2:
    print("Reversing question")
    q_i = 1
    p_i = 0
    q_j = 0
    answer_1 = hdm.answer(criterion_i=1, criterion_j=0, q_i=q_i, q_j=q_j, p_i=p_i)

    assert None not in answer_1

elif np.sum(answer_1 > 1) == 1:
    print("Reversing question")
    q_i = np.min(answer_1)
    p_i = 0
    q_j = 0
    answer_1 = hdm.answer(criterion_i=1, criterion_j=0, q_i=q_i, q_j=q_j, p_i=p_i)

    assert None not in answer_1

else:

    deducted_alpha_slope = (p_i - q_i) / (q_j - answer_1[0])
    deducted_beta_slope = (p_i - q_i) / (q_j - answer_1[1])

    slopes_alpha[1].append(deducted_alpha_slope)
    slopes_beta[1].append(deducted_beta_slope)

print("slopes:")
print(slopes_alpha, slopes_beta)


In [ ]:
q_i = 1
p_i = 0
q_j = 1

answer_1 = hdm.answer(criterion_i=0, criterion_j=1, q_i=q_i, q_j=q_j, p_i=p_i)
print(answer_1)


q_i = 1
p_i = .0
q_j = .9

answer_2 = hdm.answer(criterion_i=0, criterion_j=1, q_i=q_i, q_j=q_j, p_i=p_i)

In [ ]:
answer_2

In [ ]:
q_i = 1
p_i = 0
q_j = 1

answer_1 = hdm.answer(criterion_i=0, criterion_j=1, q_i=q_i, q_j=q_j, p_i=p_i)
print(answer_1)


q_i = 1
p_i = .0
q_j = .9

answer_2 = hdm.answer(criterion_i=0, criterion_j=1, q_i=q_i, q_j=q_j, p_i=p_i)

u_quotient_11 = (0 - 1) / (1 - answer_1[0])
u_quotient_12 = (0 - 1) / (1 - answer_1[1])


fixed_q_1 = (0 - 1) / (answer_2[0] - 1)
fixed_q_2 = (0 - 1) / (answer_2[1] - 1)
u_quotient_21 = (.9 - 1) / (answer_2[0] - 1)
u_quotient_22 = (.9 - 1) / (answer_2[1] - 1)

In [ ]:
fixed_q_1, fixed_q_2, u_quotient_21, u_quotient_22

In [ ]:
possible_slopes_1 = [
    -fixed_q_1 + u_quotient_21 * slopes_alpha[1][0],
    -fixed_q_2 + u_quotient_22 * slopes_beta[1][0],
]
possible_slopes_2 = [
    -fixed_q_2 + u_quotient_22 * slopes_alpha[1][0],
    -fixed_q_1 + u_quotient_21 * slopes_beta[1][0],
]

case_1 = np.min([
    np.abs(possible_slopes_1[0] -u_quotient_11) + np.abs(possible_slopes_1[1] -u_quotient_12), 
    np.abs(possible_slopes_1[1] -u_quotient_11) + np.abs(possible_slopes_1[0] -u_quotient_12)
])

case_2 = np.min([
    np.abs(possible_slopes_2[0] -u_quotient_11) + np.abs(possible_slopes_2[1] -u_quotient_12), 
    np.abs(possible_slopes_2[1] -u_quotient_11) + np.abs(possible_slopes_2[0] -u_quotient_12)
])

if case_1 < case_2:
    slopes_alpha[1].append(possible_slopes_1[0])
    slopes_beta[1].append(possible_slopes_1[1])

else:
    slopes_alpha[1].append(possible_slopes_2[0])
    slopes_beta[1].append(possible_slopes_2[1])

In [ ]:
slopes_alpha, slopes_beta

In [ ]:
# crit_i, crit_j, q_i, p_i, q_j

q_i = 1
p_i = 1.5
q_j = 1

answer_1 = hdm.answer(criterion_i=0, criterion_j=1, q_i=q_i, q_j=q_j, p_i=p_i)
possible_slopes_a = [
    [(answer_1[0] - 1) / (1 - 1.5) * slopes_alpha[1][0],
    (answer_1[1] - 1) / (1 - 1.5) * slopes_beta[1][0]],
    [(answer_1[1] - 1) / (1 - 1.5) * slopes_alpha[1][0],
    (answer_1[0] - 1) / (1 - 1.5) * slopes_beta[1][0]],
]


In [ ]:
# crit_i, crit_j, q_i, p_i, q_j

q_i = 1
p_i = 1.5
q_j = 2

answer_1 = hdm.answer(criterion_i=0, criterion_j=1, q_i=q_i, q_j=q_j, p_i=p_i)
possible_slopes_b = [
    [(answer_1[0] - 2) / (1 - 1.5) * slopes_alpha[1][1],
    (answer_1[1] - 2) / (1 - 1.5) * slopes_beta[1][1]],
    [(answer_1[1] - 2) / (1 - 1.5) * slopes_alpha[1][1],
    (answer_1[0] - 2) / (1 - 1.5) * slopes_beta[1][1]],
]


In [ ]:
case_1 = np.min([
    np.abs(possible_slopes_a[0][0] - possible_slopes_b[0][0]) + np.abs(possible_slopes_a[0][1] -possible_slopes_b[0][1]), 
    np.abs(possible_slopes_a[0][1] - possible_slopes_b[0][0]) + np.abs(possible_slopes_a[0][0] -possible_slopes_b[0][1]), 
])

case_2 = np.min([
    np.abs(possible_slopes_a[1][0] - possible_slopes_b[0][0]) + np.abs(possible_slopes_a[1][1] -possible_slopes_b[0][1]), 
    np.abs(possible_slopes_a[1][1] - possible_slopes_b[0][0]) + np.abs(possible_slopes_a[1][0] -possible_slopes_b[0][1]), 
])

case_3 = np.min([
    np.abs(possible_slopes_a[0][0] - possible_slopes_b[1][0]) + np.abs(possible_slopes_a[0][1] -possible_slopes_b[1][1]), 
    np.abs(possible_slopes_a[0][1] - possible_slopes_b[1][0]) + np.abs(possible_slopes_a[0][0] -possible_slopes_b[1][1]), 
])

case_4 = np.min([
    np.abs(possible_slopes_a[1][0] - possible_slopes_b[1][0]) + np.abs(possible_slopes_a[1][1] -possible_slopes_b[1][1]), 
    np.abs(possible_slopes_a[1][1] - possible_slopes_b[1][0]) + np.abs(possible_slopes_a[1][0] -possible_slopes_b[1][1]), 
])

In [ ]:
fcase = np.argmin([case_1, case_2, case_3, case_4])
if fcase == 0 or fcase == 2:
    slopes_alpha[0].append(possible_slopes_a[0][0])
    slopes_beta[0].append(possible_slopes_a[0][1])
else:
    slopes_alpha[0].append(possible_slopes_a[1][0])
    slopes_beta[0].append(possible_slopes_a[1][1])

In [ ]:
hdm.dms[0].slopes, hdm.dms[1].slopes

In [ ]:
slopes_alpha, slopes_beta

In [ ]:
slopes_alpha, slopes_beta

In [ ]:
hdm.dms[0].slopes, hdm.dms[1].slopes

In [ ]:
answer_1

In [ ]:
hdm.answer(0, 1, 1, 0, 0)

In [ ]:
(dm.answer(0, 1, 1, 0, 0)) * dm.slopes[1][0]

In [ ]:
dm.answer(0, 1, 1, 0, 0)

In [ ]:
(1 - dm.answer(0, 1, 0, 1, 1)) * dm.slopes[1][0]

In [ ]:
import pandas as pd